# Inference

In [1]:
import os
import shutil

try:
    import sam2
    model = 'sam2'
except ImportError:
    model = 'sam'

In [2]:
# scene_name = "scene0000_02"
# prompt_path = f"/home/kuzhum/sam/promptable-3d-object-segmentation/SAMPro3D/61_points.ply"
# visualize = True
# experiment_name = "guitar"
# sam2_video = False

# if model == 'sam':
#     sam2_video = False

# # Parameters
# negative_background = True
# segmentation_threshold = 0.0

In [3]:
scene_name = "scene0008_00"
prompt_path = f"/home/rpartsey/code/eai/SAMPro3D-fork/data/scannet/scans/scene0008_00/scene0008_00_sofa_chair.ply"
visualize = True
experiment_name = "sofa_chair"
sam2_video = False
sam2_reverse = True
frame_interval = 10

if model == 'sam':
    frame_interval = False
    sam2_video = False
    sam2_reverse = False

if not sam2_video:
    frame_interval = False
    sam2_reverse = False

# Parameters
negative_background = True
segmentation_threshold = 0.0

In [4]:
path_to_scripts = "/home/kuzhum/sam/promptable-3d-object-segmentation/SAMPro3D/scripts"
os.chdir(path_to_scripts)

device = "cuda:0"

# dataset params
dataset_dir = "/home/rpartsey/code/eai/SAMPro3D-fork/data/scannet/scans"

# experiment params
experiments_dir = "/home/kuzhum/sam/promptable-3d-object-segmentation/SAMPro3D/experiments"

output_folder = f"{experiments_dir}/{scene_name}/{experiment_name}_{model}{'video' if sam2_video else ''}{'reverse' if sam2_reverse else ''}{'_' + str(frame_interval) if frame_interval else ''}_{segmentation_threshold}{'_neg' if negative_background else ''}"

# If there is no variable prompt_path, the prompt will be generated from the scene_name and experiment_name
sam_output_path =  f"{output_folder}/sam_output"
sampro3d_predictions =  f"{output_folder}/sampro3d_predictions"
output_vis_path = f"{output_folder}/visualization"
experiments_path = f"../experiments"

In [ ]:
!python 3d_prompt_proposal.py \
    --data_path {dataset_dir} \
    --scene_name {scene_name} \
    --prompt_path {prompt_path} \
    --prompt_name {experiment_name} \
    --experiments_path {experiments_path} \
    --device {device} \
    {"--sam2" if model == "sam2" else ""} \
    {"--sam2_video" if sam2_video else ""} \
    {"--sam2_reverse" if sam2_reverse else ""} \
    {"--frame_interval " + str(frame_interval) if frame_interval else ""}

# If folder exists, remove it
if os.path.exists(f"{output_folder}"):
    shutil.rmtree(f"{output_folder}")
shutil.move(f"{experiments_dir}/{scene_name}/{experiment_name}", f"{output_folder}")

!python main.py \
    --data_path {dataset_dir} \
    --scene_name {scene_name} \
    --prompt_path {prompt_path} \
    --sam_output_path {sam_output_path} \
    --pred_path {sampro3d_predictions} \
    --output_vis_path {output_vis_path} \
    --device {device} \
    {"--sam2" if model == "sam2" else ""} \
    {"--sam2_video" if sam2_video else ""} \
    {"--neg_bg" if negative_background else ""} \
    --segmentation_threshold {segmentation_threshold}


# Visualisation

In [6]:
# change dir
import os
os.chdir('/home/kuzhum/sam/promptable-3d-object-segmentation/SAMPro3D/scripts')

In [7]:
import numpy as np
import os
from utils.vis_utils import create_visualization_video_with_opencv

In [ ]:
# Generate random colors for each object
colors = [np.random.randint(0, 256, 3).tolist() for _ in range(1000)]  # BGR format
object_names = [name for name in os.listdir(f"{experiments_dir}/{scene_name}") if f"_{model}{'video' if sam2_video else ''}{'reverse' if sam2_reverse else ''}_" in name]
print(f"Object names: {object_names}")

# Get number of frames in the scene
num_frames = len(os.listdir(f"{dataset_dir}/{scene_name}/color"))

if visualize:
    create_visualization_video_with_opencv(
        start_frame_idx=0,
        end_frame_idx=num_frames,
        output_video_path=f"{scene_name}_{model}{'video' if sam2_video else ''}{'reverse' if sam2_reverse else ''}.mp4",
        frame_size=(640, 480),
        colors=colors,
        dataset_dir=dataset_dir, 
        scene_name=scene_name, 
        experiments_dir=experiments_dir,
        object_names=object_names,
        model=model,
    )
